# Fine-Tune LLaVA-OneVision 7B on Eight Benchmark Types

This notebook builds one balanced eight-task mixture, fine-tunes
`llava-hf/llava-onevision-qwen2-7b-ov-hf` with QLoRA, and evaluates the saved
adapter on the same benchmark configuration as the baseline notebook.

| Type | Task | Benchmark | Evaluation split |
|---|---|---|---|
| `L` | Labeling | `fashion_mnist` | `test` |
| `A` | Visual question answering | `docvqa` | `validation` |
| `B` | Bounding-box detection | `openimages_v4_detection` | `validation` |
| `C` | Captioning | `mscoco_caption` | `test` |
| `E` | Editing prompt reconstruction | `magicbrush` | `train` |
| `G` | Generation prompt reconstruction | `diffusiondb` | `train` |
| `P` | Preference | `pick_a_pic` | `train` |
| `R` | Rating | `tad66k` | `train` |


## 1. Environment

In [ ]:
from pathlib import Path
import getpass
import json
import os
import sys

candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (path for path in candidate_roots if (path / "models").is_dir() and (path / "benchmarks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from this repository or one of its subdirectories.")
REPO_ROOT = REPO_ROOT.resolve()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

requirements_path = REPO_ROOT / "requirements.txt"
fine_tuning_requirements = REPO_ROOT / "fine_tuning" / "requirements.txt"
%pip install -q -r "$requirements_path"
%pip install -q -r "$fine_tuning_requirements"

from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
if not HF_TOKEN:
    entered_token = getpass.getpass("HF token (press Enter for anonymous access): ").strip()
    if entered_token:
        HF_TOKEN = entered_token
        os.environ["HF_TOKEN"] = entered_token
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

print("Repository:", REPO_ROOT)

## 2. Configuration

In [ ]:
MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
EVAL_SAMPLES_PER_TYPE = 100
USE_4BIT = True
OVERWRITE = False

BENCHMARK_SPECS = {
    "L": {"name": "fashion_mnist", "eval_split": "test"},
    "A": {"name": "docvqa", "eval_split": "validation"},
    "B": {"name": "openimages_v4_detection", "eval_split": "validation"},
    "C": {"name": "mscoco_caption", "eval_split": "test"},
    "E": {"name": "magicbrush", "eval_split": "train"},
    "G": {"name": "diffusiondb", "eval_split": "train"},
    "P": {"name": "pick_a_pic", "eval_split": "train"},
    "R": {"name": "tad66k", "eval_split": "train"},
}

# DocVQA exposes 1,286 usable validation images. Reserve 100 for evaluation
# and 200 for validation, leaving at most 986; use 980 for a small margin.
TRAIN_EXAMPLES_PER_TYPE = 980
VALIDATION_EXAMPLES_PER_TYPE = 200
EPOCHS = 2
LEARNING_RATE = "1e-4"
GRADIENT_ACCUMULATION_STEPS = 16
LORA_RANK = 32
SEED = 42
RUN_EXPORT = True
RUN_TRAINING = True

TRAINING_SPECS = {
    "L": {"benchmark": "fashion_mnist", "train_split": "train"},
    "A": {"benchmark": "docvqa", "train_split": "validation"},
    "B": {"benchmark": "openimages_v4_detection", "train_split": "train"},
    "C": {"benchmark": "mscoco_caption", "train_split": "train"},
    "E": {"benchmark": "magicbrush", "train_split": "train"},
    "G": {"benchmark": "diffusiondb", "train_split": "train"},
    "P": {"benchmark": "pick_a_pic", "train_split": "train"},
    "R": {"benchmark": "tad66k", "train_split": "train"},
}

DATA_ROOT = REPO_ROOT / "fine_tuning" / "data" / "llava_onevision_eight_types"
COMBINED_DIR = DATA_ROOT / "combined"
ADAPTER_DIR = REPO_ROOT / "fine_tuning" / "output" / "llava-onevision-qwen2-7b-eight-types-lora"
OUTPUT_DIR = REPO_ROOT / "results" / "llava_onevision_7b_eight_types_finetuned"
for path in (DATA_ROOT, COMBINED_DIR, ADAPTER_DIR.parent, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

## 3. Export balanced training and validation data

For same-split datasets, `--skip-examples` reserves the initial evaluation-sized slice before export.

In [ ]:
import subprocess

def run_command(command):
    command = [str(value) for value in command]
    print(" ".join(command), flush=True)
    subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
if RUN_EXPORT:
    print("Export enabled")
else:
    print("Export disabled; existing manifests will be used.")

In [ ]:
if RUN_EXPORT:
    for task_type, spec in TRAINING_SPECS.items():
        benchmark_dir = DATA_ROOT / f"{task_type}_{spec['benchmark']}"
        train_manifest = benchmark_dir / "train_manifest.jsonl"
        validation_manifest = benchmark_dir / "validation_manifest.jsonl"
        metadata_path = benchmark_dir / "metadata.json"
        reusable = False
        if train_manifest.exists() and validation_manifest.exists() and metadata_path.exists() and not OVERWRITE:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            reusable = (
                metadata.get("train_examples") == TRAIN_EXAMPLES_PER_TYPE
                and metadata.get("validation_examples") == VALIDATION_EXAMPLES_PER_TYPE
                and metadata.get("skipped_examples") == EVAL_SAMPLES_PER_TYPE
            )
        if reusable:
            print(task_type, "reusing", benchmark_dir)
            continue
        command = [
            sys.executable,
            "fine_tuning/prepare_benchmark.py",
            "--benchmark", spec["benchmark"],
            "--train-split", spec["train_split"],
            "--train-examples", TRAIN_EXAMPLES_PER_TYPE,
            "--validation-examples", VALIDATION_EXAMPLES_PER_TYPE,
            "--label-sample-size", max(512, TRAIN_EXAMPLES_PER_TYPE + VALIDATION_EXAMPLES_PER_TYPE),
            "--skip-examples", EVAL_SAMPLES_PER_TYPE,
            "--output-dir", benchmark_dir,
        ]
        run_command(command)
else:
    print("Skipping export.")

## 4. Merge the eight manifests

In [ ]:
import random
import shutil

def merge_split(split):
    records = []
    image_root = COMBINED_DIR / "images"
    image_root.mkdir(parents=True, exist_ok=True)
    for task_type, spec in TRAINING_SPECS.items():
        source_dir = DATA_ROOT / f"{task_type}_{spec['benchmark']}"
        source_manifest = source_dir / f"{split}_manifest.jsonl"
        type_image_dir = image_root / task_type
        type_image_dir.mkdir(parents=True, exist_ok=True)
        for index, line in enumerate(source_manifest.read_text(encoding="utf-8").splitlines()):
            if not line.strip():
                continue
            record = json.loads(line)
            source_image = source_dir / record["image_path"]
            suffix = source_image.suffix or ".png"
            target_image = type_image_dir / f"{split}_{index:06d}{suffix}"
            if not target_image.exists() or OVERWRITE:
                shutil.copy2(source_image, target_image)
            record["image_path"] = target_image.relative_to(COMBINED_DIR).as_posix()
            record["task_type"] = task_type
            record["benchmark"] = spec["benchmark"]
            records.append(record)
    random.Random(f"{SEED}:{split}").shuffle(records)
    output_path = COMBINED_DIR / f"{split}_manifest.jsonl"
    output_path.write_text(
        "".join(json.dumps(record, ensure_ascii=True) + "\n" for record in records),
        encoding="utf-8",
    )
    return output_path

TRAIN_MANIFEST = merge_split("train")
VALIDATION_MANIFEST = merge_split("validation")
print(TRAIN_MANIFEST, VALIDATION_MANIFEST, sep="\n")

In [ ]:
from collections import Counter

def read_manifest(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

print("Train:", Counter(row["task_type"] for row in read_manifest(TRAIN_MANIFEST)))
print("Validation:", Counter(row["task_type"] for row in read_manifest(VALIDATION_MANIFEST)))

## 5. Train the balanced QLoRA adapter

In [ ]:
training_command = [
    sys.executable,
    "fine_tuning/train_llava_onevision_7b.py",
    "--model-id", MODEL_ID,
    "--train-manifest", TRAIN_MANIFEST,
    "--validation-manifest", VALIDATION_MANIFEST,
    "--output-dir", ADAPTER_DIR,
    "--epochs", EPOCHS,
    "--learning-rate", LEARNING_RATE,
    "--gradient-accumulation-steps", GRADIENT_ACCUMULATION_STEPS,
    "--lora-rank", LORA_RANK,
    "--seed", SEED,
    "--quantization", "4bit" if USE_4BIT else "none",
]
if RUN_TRAINING:
    run_command(training_command)
else:
    print("Training skipped. Set RUN_TRAINING = True to start it.")

## 6. Load the adapter for repository benchmarks

In [ ]:
from benchmarks import (
    DiffusionDBBenchmark,
    DocVQABenchmark,
    FashionMNISTBenchmark,
    MSCOCOCaptionBenchmark,
    MagicBrushBenchmark,
    OpenImagesV4DetectionBenchmark,
    PickAPicBenchmark,
    TAD66KBenchmark,
)

BENCHMARK_CLASSES = {
    "L": FashionMNISTBenchmark,
    "A": DocVQABenchmark,
    "B": OpenImagesV4DetectionBenchmark,
    "C": MSCOCOCaptionBenchmark,
    "E": MagicBrushBenchmark,
    "G": DiffusionDBBenchmark,
    "P": PickAPicBenchmark,
    "R": TAD66KBenchmark,
}

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaOnevisionForConditionalGeneration

from models import LlavaOnevisionQwen2_7B

def load_benchmark_model(*, adapter_dir=None, name):
    model_kwargs = {
        "device_map": "auto",
        "low_cpu_mem_usage": True,
        "torch_dtype": torch.bfloat16,
        "attn_implementation": "sdpa",
    }
    if USE_4BIT:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    processor_source = adapter_dir or MODEL_ID
    processor = AutoProcessor.from_pretrained(processor_source)
    model = LlavaOnevisionForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)
    if adapter_dir is not None:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()

    wrapper = LlavaOnevisionQwen2_7B.__new__(LlavaOnevisionQwen2_7B)
    wrapper.model_id = MODEL_ID
    wrapper.name = name
    wrapper.max_new_tokens = 64
    wrapper.stream = False
    wrapper.processor = processor
    wrapper.model = model
    return wrapper

In [ ]:
if not (ADAPTER_DIR / "adapter_config.json").is_file():
    raise FileNotFoundError(f"No trained adapter found at {ADAPTER_DIR}")
adapted_model = load_benchmark_model(
    adapter_dir=ADAPTER_DIR,
    name="llava-onevision-qwen2-7b-eight-types-lora",
)
adapted_model.name

## 7. Evaluate the fine-tuned model

In [ ]:
import pandas as pd

from runners import BenchmarkRun, ModelRun, run_benchmark_matrix

def evaluate_eight_types(model, *, model_name, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    summaries = []
    for task_type, spec in BENCHMARK_SPECS.items():
        benchmark = BENCHMARK_CLASSES[task_type](
            split=spec["eval_split"],
            streaming=True,
        )
        model.max_new_tokens = benchmark.default_max_new_tokens
        result_path = output_dir / f"{model_name}_{benchmark.name}.json"
        if result_path.exists() and not OVERWRITE:
            summaries.append(
                {
                    "type": task_type,
                    "model": model_name,
                    "benchmark": benchmark.name,
                    "status": "skipped",
                    "results_path": str(result_path),
                }
            )
            continue
        try:
            summary = run_benchmark_matrix(
                models=[ModelRun(name=model_name, model=model)],
                benchmark_runs=[
                    BenchmarkRun(
                        benchmark=benchmark,
                        num_samples=EVAL_SAMPLES_PER_TYPE,
                    )
                ],
                output_dir=output_dir,
            )[0]
            summaries.append({"type": task_type, "status": "ok", **summary})
        except Exception as exc:
            summaries.append(
                {
                    "type": task_type,
                    "model": model_name,
                    "benchmark": benchmark.name,
                    "status": "error",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
        (output_dir / "run_summary.json").write_text(
            json.dumps(summaries, indent=2),
            encoding="utf-8",
        )
        print(task_type, benchmark.name, summaries[-1]["status"], flush=True)
    return pd.DataFrame(summaries)

In [ ]:
finetuned_summary = evaluate_eight_types(
    adapted_model,
    model_name="llava-onevision-qwen2-7b-eight-types-lora",
    output_dir=OUTPUT_DIR,
)
finetuned_summary

## Notes

- Run the baseline notebook with the same `EVAL_SAMPLES_PER_TYPE`, split map,
  and `USE_4BIT` value before comparing scores.
- Fashion-MNIST, OpenImages detection, and MS COCO use evaluation splits that
  differ from their training splits.
- DocVQA and the four train-only benchmark adapters may still have overlap risk
  if an upstream adapter changes sample ordering as a function of requested
  count. Treat those comparisons as exploratory unless you provide explicitly
  disjoint source splits.
- The default freezes the vision tower and trains language/projector LoRA
  modules. Add `--train-vision-tower` only as a deliberate higher-memory
  experiment.